# Complaints classifier fine-tune — GPU run on vast.ai

Fine-tunes DistilBERT on the full 162,421-row real CFPB consumer complaints dataset
(5-class: credit_reporting, debt_collection, mortgages_and_loans, credit_card, retail_banking).

Run cells **in order, top to bottom**. Checkpoints save every epoch, so if the connection
drops or the instance restarts, you can resume without losing everything.

**Cost reminder:** this instance bills per hour while running. Stop it from the vast.ai
dashboard once you've downloaded the results at the end.

## Step 1 — Check hardware and disk

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print(subprocess.run(["df", "-h"], capture_output=True, text=True).stdout)

## Step 2 — Upload the dataset

Using Jupyter's file browser (left sidebar), create a folder `/workspace/churn_text_job/data/`
and upload `complaints_processed.csv` (99MB, from your local machine) into it. Do this now,
then run the cell below to confirm it landed correctly.

In [ ]:
import os
DATA_DIR = "/workspace/churn_text_job/data"
os.makedirs(DATA_DIR, exist_ok=True)
csv_path = os.path.join(DATA_DIR, "complaints_processed.csv")
print("File exists:", os.path.exists(csv_path))
if os.path.exists(csv_path):
    print("Size (MB):", os.path.getsize(csv_path) / 1e6)

## Step 3 — Install packages

Uses `{sys.executable} -m pip` (not plain `!pip`) since on some images that installs into a
*different* Python than the notebook kernel actually runs - same gotcha we hit on the
celebrity-doppelganger vast.ai run. Installs the CUDA build of PyTorch explicitly (cu121 -
compatible with most current NVIDIA drivers via backward compatibility; if step 4 below shows
no GPU, try changing `cu121` to `cu124` or `cu118` here and re-running).

In [ ]:
import sys
!{sys.executable} -m pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!{sys.executable} -m pip install -q transformers datasets accelerate scikit-learn pandas

## Step 4 — Confirm the GPU is actually detected

**Don't skip this** - if `torch.cuda.is_available()` is False, everything below will silently
fall back to CPU and take ~90 hours instead of a few. If it's False, try re-running step 3
with a different CUDA version in the index URL (cu118 / cu124) and restart the kernel before
re-running this cell.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Step 5 — Load and split the data

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(csv_path).dropna(subset=["narrative"])
print(f"Loaded {len(df)} rows")
print(df["product"].value_counts(normalize=True))

label_names = sorted(df["product"].unique())
label_to_id = {name: i for i, name in enumerate(label_names)}
print("Labels:", label_to_id)

df["label"] = df["product"].map(label_to_id)

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

## Step 6 — Tokenize

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["narrative"], truncation=True, padding="max_length", max_length=128)

train_ds = Dataset.from_pandas(train_df[["narrative", "label"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df[["narrative", "label"]]).map(tokenize, batched=True)

## Step 7 — Define the model and class-weighted trainer

Class weights compensate for the 56%/12%/12%/10%/8% imbalance across the 5 categories -
without this the model could just learn to over-predict the majority class.

In [ ]:
import numpy as np
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(label_names))
model.to("cuda" if torch.cuda.is_available() else "cpu")

class_counts = train_df["label"].value_counts().sort_index().values
class_weights = torch.tensor(len(train_df) / (len(label_names) * class_counts), dtype=torch.float)
print("Class weights:", class_weights.tolist())


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "macro_precision": precision_score(labels, preds, average="macro"),
        "macro_recall": recall_score(labels, preds, average="macro"),
    }


args = TrainingArguments(
    output_dir="/workspace/churn_text_job/checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to=[],
)

trainer = WeightedTrainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

## Step 8 — Train

This is the long step. Watch the eval metrics printed after each epoch. If the connection
drops, you can resume from the last saved checkpoint by re-running this cell with
`trainer.train(resume_from_checkpoint=True)` instead.

In [ ]:
trainer.train()

## Step 9 — Final evaluation and save

In [ ]:
import json

metrics = trainer.evaluate()
print(metrics)

MODEL_OUT_DIR = "/workspace/churn_text_job/model"
trainer.save_model(MODEL_OUT_DIR)
tokenizer.save_pretrained(MODEL_OUT_DIR)

with open("/workspace/churn_text_job/metrics.json", "w") as f:
    json.dump({"metrics": metrics, "label_to_id": label_to_id}, f, indent=2)

print(f"\nModel saved to {MODEL_OUT_DIR}")
print("Compare against the local TF-IDF+LogReg sanity check: accuracy 0.846, macro F1 0.818")

## Step 10 — Zip and download

Download `complaints_model.zip` via Jupyter's file browser (left sidebar → navigate to
`churn_text_job/complaints_model.zip` → right-click → Download), then drop it into
this project's `models/` folder locally.

In [ ]:
import shutil
shutil.make_archive("/workspace/churn_text_job/complaints_model", "zip", MODEL_OUT_DIR)
shutil.copy("/workspace/churn_text_job/metrics.json", "/workspace/churn_text_job/")
print(os.path.getsize("/workspace/churn_text_job/complaints_model.zip") / 1e6, "MB")

## Done

Once you've downloaded the model zip and `metrics.json`, **stop this instance** from the
vast.ai dashboard to stop the hourly billing.